<a href="https://colab.research.google.com/github/tdog9237/swot/blob/master/Latest_Qwen3_TTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗣️ Qwen3-TTS Colab

## 📄 Description

This Colab notebook runs **Qwen3-TTS-12Hz-0.6B-Base** and **Qwen3-TTS-12Hz-1.7B-Base**, powerful **multilingual, low-latency text-to-speech (TTS)** models from the **Qwen3 TTS family**.
Designed with a **universal end-to-end architecture**, Qwen3-TTS delivers **high-fidelity**, **instruction-controllable**, and **real-time streaming** speech synthesis with strong robustness to noisy or complex text inputs.

**Capabilities:**
Multilingual TTS (10 Languages), Ultra-Low-Latency Streaming (≈97ms), Instruction-Based Voice Control, Rapid 3s Voice Cloning, High-Fidelity Speech Reconstruction

---

## How to use

* Modify text and instruction variables
* Run all following cells, upload reference audio if needed, and generate speech

---

## ⚙️ Model Highlights

* 🌍 **10-language support** – Chinese, English, Japanese, Korean, German, French, Russian, Portuguese, Spanish, Italian
* ⚡ **Extreme low-latency generation** – first audio packet emitted after a single character input
* 🧠 **Instruction-aware speech synthesis** – adaptive control over tone, emotion, prosody, and speaking rate
* 🧬 **3-second rapid voice cloning** – supported by both Base models
* 🏗 **End-to-end discrete LM architecture** – avoids cascading errors of traditional TTS pipelines

---

## 🧠 Model Details

* **Models Included:** Qwen3-TTS-12Hz-0.6B-Base, Qwen3-TTS-12Hz-1.7B-Base
* **Speech Tokenizer:** Qwen3-TTS-Tokenizer-12Hz
* **Architecture:** Discrete multi-codebook LM (non-DiT)
* **Streaming Support:** Yes (streaming & non-streaming in one model)
* **Latency:** As low as ~97 ms end-to-end
* **Use Cases:** Real-time assistants, voice agents, multilingual narration, TTS fine-tuning

---

## 🔗 Resources

* **Hugging Face (1.7B):** https://huggingface.co/Qwen/Qwen3-TTS-12Hz-1.7B-Base  
* **Hugging Face (0.6B):** https://huggingface.co/Qwen/Qwen3-TTS-12Hz-0.6B-Base  
* **Official Blog:** https://qwen.ai/blog

---

## 🎙️ Explore More TTS Models

Looking for more cutting-edge voice models?
👉 Check out the full collection: [awesome-TTS-Colab](https://github.com/Troyanovsky/awesome-TTS-Colab)


## TTS/Voice Generation with Voice Cloning

In [13]:
!ngrok authtoken 2zJc5VsXcAm0pKnENT5VMddDMsD_DAV9ZV1TUwfpdWF7UiDd

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
# Server cell in Colab
import nest_asyncio, asyncio, os
from fastapi import FastAPI, Form
from fastapi.responses import FileResponse
import uvicorn
from pyngrok import ngrok
import soundfile as sf
from qwen_tts import Qwen3TTSModel  # ensure qwen-tts installed
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Patch asyncio for Colab
nest_asyncio.apply()

# -------------------------
# Settings
LLM_MODEL = "tiiuae/falcon-7b-instruct" # e.g., Llama2 or local Ollama
TTS_MODEL = "Qwen3-TTS-Base"  # your chosen Qwen model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REF_AUDIO = "/content/morgan.wav"  # always use this

# -------------------------
# Load LLM
print("Loading LLM model...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm_model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, device_map="auto")
print("LLM loaded on", DEVICE)

# Load Qwen3-TTS
print("Loading Qwen3-TTS model...")
tts_model = Qwen3TTSModel.from_pretrained(TTS_MODEL, device_map="auto")
print("TTS loaded")

# -------------------------
# FastAPI app
app = FastAPI()

@app.post("/speak")
async def speak(text: str = Form(...)):
    """Receive text, generate LLM answer, then TTS audio"""

    # 1️⃣ LLM generate answer
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    outputs = llm_model.generate(**inputs, max_new_tokens=200)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # 2️⃣ Qwen3-TTS generate voice using fixed reference
    wavs, sr = tts_model.generate_voice_clone(
        text=answer,
        language="English",
        ref_audio=REF_AUDIO,
        ref_text="Hello, this is Morgan speaking."  # dummy text for voice clone
    )

    # 3️⃣ Save audio file
    out_path = "/content/output.wav"
    sf.write(out_path, wavs[0], sr)

    # 4️⃣ Return audio file
    return FileResponse(out_path, media_type="audio/wav", filename="output.wav")

# -------------------------
# Expose via ngrok
port = 8000
public_url = ngrok.connect(port)
print("🔗 Public URL:", public_url)

# -------------------------
# Run Uvicorn asynchronously
async def start_server():
    config = uvicorn.Config(app, host="0.0.0.0", port=port)
    server = uvicorn.Server(config)
    await server.serve()

asyncio.run(start_server())


    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 
Loading LLM model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.48G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.95G [00:00<?, ?B/s]